# Installing Net2Brain

In [ ]:
# NATURALISTIC_PATH_SETUP
from pathlib import Path
import os
STIMULI = Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli'))
DATA = Path(os.environ.get('NATURALISTIC_ENCODING_DATA', '../data'))

import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from scipy.signal import resample


In [ ]:
#use conda env net2brain

In [ ]:
#!pip install -U git+https://github.com/cvai-roig-lab/Net2Brain

# Step 1: Feature Extraction

## Using `FeatureExtractor` with a model from Net2Brain

The FeatureExtractor class provides an interface for extracting features from a given model. When initializing this class, you can customize its behavior by setting various parameters:

- `model` (required): The model from which you want to extract features. Either string in combination with a netset (next parameter), or a variable with a model-type.
- `netset` (optional): The netset (collection of networks) that the model belongs to.
- `layers_to_extract` (optional): A list of layer names or indices from which you want to extract features. Default is None, indicating that all layers preset in the toolbox will be used.
- `device` (optional): The device on which to perform the computations, e.g., 'cuda' for GPU or 'cpu' for CPU. Default is None, which will use the device specified in the global PyTorch settings.
- `transforms` (optional): A list of data preprocessing transforms to apply to the input data before passing it through the model. Default is None, which uses the preset transformations
- `pretrained` (optional): A boolean flag indicating whether to use a pretrained model (if available) or to initialize the model with random weights. Default is True, which means that a pretrained model will be used if possible.

- - -


First we need to a dataset to play around with. For that we will use the dataset by [Micheal F. Bonner (2017)](https://www.pnas.org/doi/full/10.1073/pnas.1618228114), which we can download using the `load_dataset` function

In [ ]:
# from net2brain.utils.download_datasets import load_dataset


### Initating FeatureExtractor


To extract the activations of a pretrained model from a netset, you can use the FeatureExtractor class. First, you need to initialize the class by providing the name of the model and the name of the netset. You can find a suitable model and netset by exploring the taxonomy options available in the Net2Brain toolbox, as shown in the previous notebook "0_Exploring_Net2Brain". For instance, in the following example, we will use AlexNet from the standard netset.

## extract resnet50

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='ResNet50', netset='Standard', device='cuda') #cuda not working now, try dif gpu? or 'cpu'

The `extract` method computes feature extraction from an image dataset. It takes in the following parameters:

- `data_path` (required): The path to the images from which to extract the features. The images must be in JPEG or PNG format.
- `save_path` (optional): The path to save the extracted features to. If None, the folder where the features are saved is named after the current date in the format "{year}{month}{day}{hour}{minute}".
- `layers_to_extract` (optional): A list of layer names or indices from which to extract features. If None, the specified layers will be used.
- `consolidate_per_layer` (optional): The features are extracted image-wise. This is defaulted to true and will consolidate them per layer if not set to False. Defautls to True.
- `dim_reduction` (optional): Type of dimensionality reduction (For now: SRP) for extracted features. Defaults to None.
- `n_components` (optonal): Number of components for dimensionality reduction. Defaults to 50.

### extract from all frames
this took like ~150gb memory i think to concatenate things and i sued teslav100 gpu for extraction

In [ ]:
import os
import shutil
from math import ceil

# Input directory
source_dir = f'../data/DM_frames_all/'

# Create destination directories
destination_dirs = [f'../data/DM_frames_all{i+1}/' for i in range(5)]
for dest_dir in destination_dirs:
    os.makedirs(dest_dir, exist_ok=True)

# Get all files sorted
all_files = sorted(f for f in os.listdir(source_dir) if f.endswith('.jpg'))

# Split into chunks
chunk_size = ceil(len(all_files) / 5)
for i, dest_dir in enumerate(destination_dirs):
    start_idx = i * chunk_size
    end_idx = min(start_idx + chunk_size, len(all_files))
    for file_name in all_files[start_idx:end_idx]:
        shutil.move(os.path.join(source_dir, file_name), os.path.join(dest_dir, file_name))

print("Files have been split into 5 directories.")


In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import os

for ii in ['2','3','4','5']:
    
    stim='DM'
    stimuli_path = f'../data/{stim}_frames_all{ii}/'
    save_path = f'../data/{stim}_frames_all{ii}_resnet50/'
    
    if not os.path.isdir(save_path):
        os.mkdir(save_path) 
        
    #relu, maxpool, block1, block2, block3, block4, avgpool
    layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']
    
    fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract)  #, consolidate_per_layer=False)

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import os

for stim in ['friends_s01e01a','friends_s01e01b','friends_s01e02a','friends_s01e02b']:
    stimuli_path = f'../data/{stim}_frames/'
    save_path = f'../data/{stim}_frames_resnet50/'
    
    if not os.path.isdir(save_path):
        os.mkdir(save_path) 
        
    #relu, maxpool, block1, block2, block3, block4, avgpool
    layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']
    
    fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract)  #, consolidate_per_layer=False)

__Net2Brain__ chooses by default from which layers of the model to extract the features from. You can inspect which layers are selected by default by calling the `layers_to_extract` attribute:

In [ ]:
fx.layers_to_extract

These are not all the layers that **can** be extracted. If you want to see all the layers that can possibly be extracted you you call `get_all_layers()`.

In [ ]:
fx.get_all_layers()

If you wish to change the layers to be extracted you can add it to the `extract` function like with the parameter 
```
fx.extract(..., layers_to_extract=[your_layers])
```

## extract yolo labels

In [ ]:
# from net2brain.taxonomy import show_all_architectures
# show_all_architectures()


In [ ]:
from net2brain.taxonomy import show_all_netsets
from net2brain.taxonomy import print_netset_models
from net2brain.taxonomy import show_taxonomy

print_netset_models('Yolo')


In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='yolov5x', netset='Yolo', device='cpu') #cuda worked with 1080ti

In [ ]:
fx.get_all_layers()

#### normal

In [ ]:
from net2brain.feature_extraction import FeatureExtractor

stim='DM'
stimuli_path = f'../data/{stim}_frames/'
save_path = f'../data/{stim}_frames_yolov5x/'

if not os.path.isdir(save_path):
    os.mkdir(save_path) 
    
#relu, maxpool, block1, block2, block3, block4, avgpool
layers_to_extract=['model.model.model.24']

fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract)  #, consolidate_per_layer=False)

#### frames all

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='yolov5x', netset='Yolo', device='cuda') #cuda worked with 1080ti


stim='DM'
stimuli_path = f'../data/{stim}_frames_all/'
save_path = f'../data/{stim}_frames_all_yolov5x/'

if not os.path.isdir(save_path):
    os.mkdir(save_path) 
    
#relu, maxpool, block1, block2, block3, block4, avgpool
layers_to_extract=['model.model.model.24']

fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract)  #, consolidate_per_layer=False)

### New Yolo11 features parse

In [ ]:
from pathlib import Path
import os
import numpy as np
import os

# Path to your YOLO text outputs
txt_dir = str(Path(os.environ.get('NATURALISTIC_ENCODING_YOLO_DIR', '../data/yolo_labels')) / 'yolo_DM' / 'labels')

# Number of frames and number of classes (80 for COCO dataset)
num_frames = 17995
num_classes = 80

# Initialize a 17998 x 80 NumPy array filled with zeros
confidence_matrix = np.zeros((num_frames, num_classes))

# Iterate over each frame's text file (DM_1.txt to DM_17998.txt)
for i in range(1, num_frames + 1):
    # Construct the filename (e.g., DM_1.txt, DM_2.txt, ..., DM_17998.txt)
    txt_file = os.path.join(txt_dir, f"DM_{i}.txt")
    
    # Check if the file exists
    if os.path.exists(txt_file):
        # Open and read the file
        with open(txt_file, 'r') as file:
            # Each line corresponds to a detection
            for line in file:
                # Extract class_id and confidence from the line
                data = line.strip().split()
                class_id = int(data[0])  # First value is the class ID
                confidence = float(data[5])  # Sixth value is the confidence
                
                # Update the confidence_matrix at the corresponding frame and class
                if confidence > confidence_matrix[i - 1, class_id]:
                    confidence_matrix[i - 1, class_id] = confidence
# Now, confidence_matrix is a 17998 x 80 array
# Save the result to a file if needed
np.save('../data/features/yolo11x_DM_all.npy', confidence_matrix)

import numpy as np
import scipy.signal as signal
def filter_data(confidence_matrix):
    original_size = 17995
    new_size = 750
    D = original_size / new_size
    # Normalized cutoff frequency (as a fraction of the Nyquist frequency)
    cutoff = 1 / (2 * D)
    # Design an FIR filter
    numtaps = 101  # Number of filter taps, controls sharpness of the cutoff
    fir_filter = signal.firwin(numtaps, cutoff)
    filtered_data = signal.lfilter(fir_filter, [1.0], confidence_matrix, axis=0)
    return filtered_data


confidence_matrix_resamp = resample(confidence_matrix, 750, axis=0) #resample to 1hz for now
np.save('../data/features/yolo11x_DM_resamp.npy', confidence_matrix_resamp)
confidence_matrix_lpf=filter_data(confidence_matrix)
confidence_matrix_lpf_resamp = resample(confidence_matrix_lpf, 750, axis=0) #resample to 1hz for now
np.save('../data/features/yolo11x_DM_lpfresamp.npy', confidence_matrix_lpf_resamp)


### SPLIT IT UP INTO CLASSES

confidence_matrix_resamp

categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}

# Create an array to hold the sum of each category
confidence_matrix_resamp_sums = np.zeros((confidence_matrix_resamp.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    confidence_matrix_resamp_sums[:, i] = np.max(confidence_matrix_resamp[:, indices], axis=1)

np.save('../data/features/yolo11x8_DM_resamp.npy', confidence_matrix_resamp_sums)




confidence_matrix_resamp

categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}

# Create an array to hold the sum of each category
confidence_matrix_resamp_sums = np.zeros((confidence_matrix_lpf_resamp.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    confidence_matrix_resamp_sums[:, i] = np.max(confidence_matrix_lpf_resamp[:, indices], axis=1)

np.save('../data/features/yolo11x8_DM_lpfresamp.npy', confidence_matrix_resamp_sums)


#### New binarizing method

In [ ]:
from pathlib import Path
import os
import numpy as np
import os
import cv2


stim='DM'
TR=0.8

print(stim)
txt_dir = str(Path(os.environ.get('NATURALISTIC_ENCODING_YOLO_DIR', '../data/yolo_labels')) / f'yolo_{stim}' / 'labels')

video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_DATA', '../data')) / f'{stim}.mp4')


video = cv2.VideoCapture(video_path)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
fps = video.get(cv2.CAP_PROP_FPS)
video.release()

# Path to your YOLO text outputs

# Number of frames and number of classes (80 for COCO dataset)
num_frames = total_frames
num_classes = 80

# Initialize a 17998 x 80 NumPy array filled with zeros
confidence_matrix = np.zeros((num_frames, num_classes))



# Iterate over each frame's text file (DM_1.txt to DM_17998.txt)
for i in range(1, num_frames + 1):
    # Construct the filename (e.g., DM_1.txt, DM_2.txt, ..., DM_17998.txt)
    txt_file = os.path.join(txt_dir, f"{stim}_{i}.txt")
    
    # Check if the file exists
    if os.path.exists(txt_file):
        # Open and read the file
        with open(txt_file, 'r') as file:
            # Each line corresponds to a detection
            for line in file:
                # Extract class_id and confidence from the line
                data = line.strip().split()
                class_id = int(data[0])  # First value is the class ID
                confidence = float(data[5])  # Sixth value is the confidence
                
                # Update the confidence_matrix at the corresponding frame and class
                if confidence > confidence_matrix[i - 1, class_id]:
                    confidence_matrix[i - 1, class_id] = confidence
# Now, confidence_matrix is a 17998 x 80 array
# Save the result to a file if needed
np.save(f'../data/features/yolo11x_{stim}_all.npy', confidence_matrix)


# make empty array of shape 472x8
num_TRs=int(np.round(total_frames/fps/TR))
num_high_classes=8

yolo11x_conf = np.zeros((num_TRs, 80))
yolo11x_binary = np.zeros((num_TRs, 80))
yolo11x8_conf = np.zeros((num_TRs, num_high_classes))
yolo11x8_binary = np.zeros((num_TRs, num_high_classes))

categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}



#loop through windows
for TR_i in np.arange(num_TRs):
    start_ind=int(np.round((TR_i*fps*TR)))
    end_ind=int(np.round(((TR_i+1)*fps*TR)))
    windowed=confidence_matrix[start_ind:end_ind,:]
    yolo11x_conf[TR_i,:]=np.max(windowed,axis=0)

np.save(f'../data/features/yolo11x_{stim}_conf.npy', yolo11x_conf)

# Create an array to hold the sum of each category
yolo11x8_conf = np.zeros((num_TRs, num_high_classes))
# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    yolo11x8_conf[:, i] = np.max(yolo11x_conf[:, indices], axis=1)

np.save(f'../data/features/yolo11x8_{stim}_conf.npy', yolo11x8_conf)
yolo11x_binary = np.where(yolo11x_conf > 0.8, 1, 0)
yolo11x8_binary = np.where(yolo11x8_conf > 0.8, 1, 0)
np.save(f'../data/features/yolo11x8_{stim}_binary.npy', yolo11x8_binary)
np.save(f'../data/features/yolo11x_{stim}_binary.npy', yolo11x_binary)



#np.save(f'../data/features/yolo11x8_{stim}_resamp.npy', confidence_matrix_resamp_sums)


In [ ]:
confidence_matrix

In [ ]:
import matplotlib.pyplot as plt
plt.plot(yolo11x8_binary)

In [ ]:
from pathlib import Path
import os
import glob
import nibabel as nb
task='s01e02a'
sub='01'

fmriprep_folder = str(Path(os.environ.get('NATURALISTIC_ENCODING_FMRIPREP_DIR', '../data/cneuromod_fmriprep')) / 'friends')
pattern=f'{fmriprep_folder}/sub-{sub}/sub-{sub}_ses-*_task-{task}_space-fsLR_den-91k_bold_cleaned_smoothed.dtseries.nii'
print(pattern)
print(glob.glob(pattern))

im_file = glob.glob(pattern)[0]
img = nb.load(im_file)
img_y = img.get_fdata()
print(f'loaded brain data')


In [ ]:
img_y.shape

#### for friends as well NEW OLDD

In [ ]:
from pathlib import Path
import os
import numpy as np
import os
import cv2

stim='friends_s01e02b'
txt_dir = str(Path(os.environ.get('NATURALISTIC_ENCODING_YOLO_DIR', '../data/yolo_labels')) / f'yolo_results_{stim}' / 'labels')

video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends.stimuli' / 's1' / f'{stim}.mkv')
video = cv2.VideoCapture(video_path)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
video.release()
# Path to your YOLO text outputs

# Number of frames and number of classes (80 for COCO dataset)
num_frames = total_frames
num_classes = 80

# Initialize a 17998 x 80 NumPy array filled with zeros
confidence_matrix = np.zeros((num_frames, num_classes))

# Iterate over each frame's text file (DM_1.txt to DM_17998.txt)
for i in range(1, num_frames + 1):
    # Construct the filename (e.g., DM_1.txt, DM_2.txt, ..., DM_17998.txt)
    txt_file = os.path.join(txt_dir, f"{stim}_{i}.txt")
    
    # Check if the file exists
    if os.path.exists(txt_file):
        # Open and read the file
        with open(txt_file, 'r') as file:
            # Each line corresponds to a detection
            for line in file:
                # Extract class_id and confidence from the line
                data = line.strip().split()
                class_id = int(data[0])  # First value is the class ID
                confidence = float(data[5])  # Sixth value is the confidence
                
                # Update the confidence_matrix at the corresponding frame and class
                if confidence > confidence_matrix[i - 1, class_id]:
                    confidence_matrix[i - 1, class_id] = confidence
# Now, confidence_matrix is a 17998 x 80 array
# Save the result to a file if needed
np.save(f'../data/features/yolo11x_{stim}_all.npy', confidence_matrix)

import numpy as np
import scipy.signal as signal
def filter_data(confidence_matrix):
    original_size = num_frames
    new_size = 472
    D = original_size / new_size
    # Normalized cutoff frequency (as a fraction of the Nyquist frequency)
    cutoff = 1 / (2 * D)
    # Design an FIR filter
    numtaps = 101  # Number of filter taps, controls sharpness of the cutoff
    fir_filter = signal.firwin(numtaps, cutoff)
    filtered_data = signal.lfilter(fir_filter, [1.0], confidence_matrix, axis=0)
    return filtered_data

new_size = 472
confidence_matrix_resamp = resample(confidence_matrix, new_size, axis=0) #resample to 1hz for now
np.save(f'../data/features/yolo11x_{stim}_resamp.npy', confidence_matrix_resamp)
confidence_matrix_lpf=filter_data(confidence_matrix)
confidence_matrix_lpf_resamp = resample(confidence_matrix_lpf, new_size, axis=0) #resample to 1hz for now
np.save(f'../data/features/yolo11x_{stim}_lpfresamp.npy', confidence_matrix_lpf_resamp)


### SPLIT IT UP INTO CLASSES

confidence_matrix_resamp

categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}

# Create an array to hold the sum of each category
confidence_matrix_resamp_sums = np.zeros((confidence_matrix_resamp.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    confidence_matrix_resamp_sums[:, i] = np.max(confidence_matrix_resamp[:, indices], axis=1)

np.save(f'../data/features/yolo11x8_{stim}_resamp.npy', confidence_matrix_resamp_sums)




# confidence_matrix_resamp

#     'Person': [0],
#     'Vehicle': list(range(1,9)),
#     'Outdoor': list(range(9,14)),
#     'Animal': list(range(14, 24)),
#     'Accessories': list(range(24,29)),
#     'Sports': list(range(29,39)),
#     'Food': list(range(39,56)),
#     'Household': list(range(56,80))
# }

# # Create an array to hold the sum of each category

# # Efficiently sum the values for each category
# for i, indices in enumerate(categories.values()):

# np.save(f'../data/features/yolo11x8_{stim}_lpfresamp.npy', confidence_matrix_resamp_sums)


#### new method binarizing 80%

In [ ]:
from pathlib import Path
import os
import numpy as np
import os
import cv2


for episode in range(3, 8):  # This will loop from 2 to 7 inclusive
    for variant in ['a', 'b']:
        stim = f"friends_s01e{episode:02d}{variant}"  # Zero-padded episode numbers
        print(stim)
        txt_dir = str(Path(os.environ.get('NATURALISTIC_ENCODING_YOLO_DIR', '../data/yolo_labels')) / f'yolo_results_{stim}' / 'labels')
        
        video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends.stimuli' / 's1' / f'{stim}.mkv')
        video = cv2.VideoCapture(video_path)
        total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = video.get(cv2.CAP_PROP_FPS)
        video.release()
        
        # Path to your YOLO text outputs
        
        # Number of frames and number of classes (80 for COCO dataset)
        num_frames = total_frames
        num_classes = 80
        
        # Initialize a 17998 x 80 NumPy array filled with zeros
        confidence_matrix = np.zeros((num_frames, num_classes))
        
        
        
        # Iterate over each frame's text file (DM_1.txt to DM_17998.txt)
        for i in range(1, num_frames + 1):
            # Construct the filename (e.g., DM_1.txt, DM_2.txt, ..., DM_17998.txt)
            txt_file = os.path.join(txt_dir, f"{stim}_{i}.txt")
            
            # Check if the file exists
            if os.path.exists(txt_file):
                # Open and read the file
                with open(txt_file, 'r') as file:
                    # Each line corresponds to a detection
                    for line in file:
                        # Extract class_id and confidence from the line
                        data = line.strip().split()
                        class_id = int(data[0])  # First value is the class ID
                        confidence = float(data[5])  # Sixth value is the confidence
                        
                        # Update the confidence_matrix at the corresponding frame and class
                        if confidence > confidence_matrix[i - 1, class_id]:
                            confidence_matrix[i - 1, class_id] = confidence
        # Now, confidence_matrix is a 17998 x 80 array
        # Save the result to a file if needed
        np.save(f'../data/features/yolo11x_{stim}_all.npy', confidence_matrix)
        
        
        # make empty array of shape 472x8
        TR=1.49
        num_TRs=int(np.round(total_frames/fps/TR))
        num_high_classes=8
        
        yolo11x_conf = np.zeros((num_TRs, 80))
        yolo11x_binary = np.zeros((num_TRs, 80))
        yolo11x8_conf = np.zeros((num_TRs, num_high_classes))
        yolo11x8_binary = np.zeros((num_TRs, num_high_classes))
        
        categories = {
            'Person': [0],
            'Vehicle': list(range(1,9)),
            'Outdoor': list(range(9,14)),
            'Animal': list(range(14, 24)),
            'Accessories': list(range(24,29)),
            'Sports': list(range(29,39)),
            'Food': list(range(39,56)),
            'Household': list(range(56,80))
        }
        
        
        
        #loop through windows
        for TR_i in np.arange(num_TRs):
            start_ind=int(np.round((TR_i*fps*TR)))
            end_ind=int(np.round(((TR_i+1)*fps*TR)))
            windowed=confidence_matrix[start_ind:end_ind,:]
            yolo11x_conf[TR_i,:]=np.max(windowed,axis=0)
        
        np.save(f'../data/features/yolo11x_{stim}_conf.npy', yolo11x_conf)
        
        # Create an array to hold the sum of each category
        yolo11x8_conf = np.zeros((num_TRs, num_high_classes))
        # Efficiently sum the values for each category
        for i, indices in enumerate(categories.values()):
            yolo11x8_conf[:, i] = np.max(yolo11x_conf[:, indices], axis=1)
        
        np.save(f'../data/features/yolo11x8_{stim}_conf.npy', yolo11x8_conf)
        yolo11x_binary = np.where(yolo11x_conf > 0.8, 1, 0)
        yolo11x8_binary = np.where(yolo11x8_conf > 0.8, 1, 0)
        np.save(f'../data/features/yolo11x8_{stim}_binary.npy', yolo11x8_binary)
        np.save(f'../data/features/yolo11x_{stim}_binary.npy', yolo11x_binary)
        
        
        
        #np.save(f'../data/features/yolo11x8_{stim}_resamp.npy', confidence_matrix_resamp_sums)


In [ ]:
yolo11x8_conf.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(confidence_matrix[:,5])

## explore features

In [ ]:
stim='DM'
save_path = f'../data/{stim}_frames_resnet50/'

#print the
emb_list = glob.glob(f'{save_path}*.npz')

layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']

print('ResNet50 frame embeddings')
# for emb_f in emb_list:
#     print(emb['frame_0000.jpg'].shape, os.path.splitext(os.path.basename(emb_f))[0])
for layer in layers_to_extract:
    emb = np.load(f'{save_path}{layer}.npz')
    print(emb['frame_0000.jpg'].shape, layer)


In [ ]:
emb['frame_0000.jpg'].shape

In [ ]:
stim='DM'
save_path = f'../data/{stim}_frames_yolov5x/'

#print the
emb_list = glob.glob(f'{save_path}*.npz')

layers_to_extract=['model.model.model.24']

print('yolov5x frame embeddings shape')
# for emb_f in emb_list:
#     print(emb['frame_0000.jpg'].shape, os.path.splitext(os.path.basename(emb_f))[0])
for layer in layers_to_extract:
    emb = np.load(f'{save_path}{layer}.npz')
    print(emb['frame_0000.jpg'].shape, layer)


#### get yolo features [OLD]

In [ ]:
def load_video_features(stim,all_layers):
    save_path = f'../data/{stim}_frames_yolov5x/'
    #print('ResNet50 frame embeddings')
    # for emb_f in emb_list:
    #     print(emb['frame_0000.jpg'].shape, os.path.splitext(os.path.basename(emb_f))[0])
    X=[]
    for layer in all_layers:
        X_layer=[]
        emb = np.load(f'{save_path}{layer}.npz')
        for k in list(emb.keys()):
            X_layer.append(emb[k])
        #X.append(  transformer.fit_transform(  np.array(X_layer)[:(-1*delay),:]  )  )
        X.append(  np.array(X_layer)  )

    return(X)
yolos=load_video_features('DM',['model.model.model.24'])
yolos=np.squeeze(yolos[0])

In [ ]:
yolos.shape

In [ ]:
import numpy as np

# Assuming `yolos` is the numpy array with shape (750, 3087, 85)

# Extract objectness scores (index 4)
objectness_scores = yolos[:, :, 4]  # Shape: (750, 3087)

# Extract class probabilities (index 5 to 84)
class_probs = yolos[:, :, 5:]  # Shape: (750, 3087, 80)

# Multiply class probabilities by the objectness score for each detection
weighted_class_probs = class_probs * objectness_scores[:, :, np.newaxis]  # Shape: (750, 3087, 80)

class_scores = np.sum(weighted_class_probs, axis=1)  # Shape: (750, 80)

# If you want normalized scores, divide by the sum of objectness scores for each frame
sum_objectness_scores = np.sum(objectness_scores, axis=1, keepdims=True)  # Shape: (750, 1)
normalized_class_scores = class_scores / sum_objectness_scores  # Shape: (750, 80)

# Now `class_scores` is the class score for each frame (750 frames, 80 classes)
# and `normalized_class_scores` are the normalized class scores for each frame


categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}

# Create an array to hold the sum of each category
category_sums = np.zeros((class_scores.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    category_sums[:, i] = np.sum(class_scores[:, indices], axis=1)



# Create an array to hold the sum of each category
category_sums_norm = np.zeros((normalized_class_scores.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    category_sums_norm[:, i] = np.sum(normalized_class_scores[:, indices], axis=1)


np.save('../data/features/DM_yolov5x_class_scores80.npy',class_scores)
np.save('../data/features/DM_yolov5x_class_scores80_norm.npy',normalized_class_scores)
np.save('../data/features/DM_yolov5x_class_scores8.npy',category_sums)
np.save('../data/features/DM_yolov5x_class_scores8_norm.npy',category_sums_norm)

In [ ]:
import pilot
p=pilot.load_features('yolov5x_class_scores8_norm')

In [ ]:
import matplotlib.pyplot as plt
plt.plot(p[0][:,0])


# The result is stored in category_sums with shape (750, 8)


#### get yolo features ALL

In [ ]:
def load_video_features(stim,all_layers):
    save_path = f'../data/{stim}_frames_all_yolov5x/'
    #print('ResNet50 frame embeddings')
    # for emb_f in emb_list:
    #     print(emb['frame_0000.jpg'].shape, os.path.splitext(os.path.basename(emb_f))[0])
    X=[]
    for layer in all_layers:
        X_layer=[]
        emb = np.load(f'{save_path}{layer}.npz')
        for k in list(emb.keys()):
            X_layer.append(emb[k])
        #X.append(  transformer.fit_transform(  np.array(X_layer)[:(-1*delay),:]  )  )
        X.append(  np.array(X_layer)  )

    return(X)
yolos=load_video_features('DM',['model.model.model.24'])
yolos=np.squeeze(yolos[0])

In [ ]:
yolos.shape

In [ ]:
import numpy as np

# Assuming `yolos` is the numpy array with shape (750, 3087, 85)

# Extract objectness scores (index 4)
objectness_scores = yolos[:, :, 4]  # Shape: (750, 3087)

# Extract class probabilities (index 5 to 84)
class_probs = yolos[:, :, 5:]  # Shape: (750, 3087, 80)

# Multiply class probabilities by the objectness score for each detection
weighted_class_probs = class_probs * objectness_scores[:, :, np.newaxis]  # Shape: (750, 3087, 80)

class_scores = np.sum(weighted_class_probs, axis=1)  # Shape: (750, 80)

# If you want normalized scores, divide by the sum of objectness scores for each frame
sum_objectness_scores = np.sum(objectness_scores, axis=1, keepdims=True)  # Shape: (750, 1)
normalized_class_scores = class_scores / sum_objectness_scores  # Shape: (750, 80)

# Now `class_scores` is the class score for each frame (750 frames, 80 classes)
# and `normalized_class_scores` are the normalized class scores for each frame


categories = {
    'Person': [0],
    'Vehicle': list(range(1,9)),
    'Outdoor': list(range(9,14)),
    'Animal': list(range(14, 24)),
    'Accessories': list(range(24,29)),
    'Sports': list(range(29,39)),
    'Food': list(range(39,56)),
    'Household': list(range(56,80))
}

# Create an array to hold the sum of each category
category_sums = np.zeros((class_scores.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    category_sums[:, i] = np.sum(class_scores[:, indices], axis=1)



# Create an array to hold the sum of each category
category_sums_norm = np.zeros((normalized_class_scores.shape[0], len(categories)))

# Efficiently sum the values for each category
for i, indices in enumerate(categories.values()):
    category_sums_norm[:, i] = np.sum(normalized_class_scores[:, indices], axis=1)


np.save('../data/features/DM_yolov5x_class_scores80_all.npy',class_scores)
np.save('../data/features/DM_yolov5x_class_scores80_norm_all.npy',normalized_class_scores)
np.save('../data/features/DM_yolov5x_class_scores8_all.npy',category_sums)
np.save('../data/features/DM_yolov5x_class_scores8_norm_all.npy',category_sums_norm)

## get pliers features saved

In [ ]:
from pathlib import Path
import os

pliers_all = pd.read_csv(Path(os.environ.get('NATURALISTIC_ENCODING_DATA', '../data')) / 'features' / 'DM_pliers_all.csv')
clarifai=np.asanyarray(pliers_all.iloc[:,5:71])
clarifai = resample(clarifai, 750, axis=0) #resample to 1hz for now
np.save('../data/features/DM_clarifai.npy',clarifai)

In [ ]:
clarifai.shape

## try dimensionality reduction features, what shape are they?

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='ResNet50', netset='Standard', device='cpu') #cuda not working now, try dif gpu?
from net2brain.feature_extraction import FeatureExtractor
import os

stim='DM'
stimuli_path = f'../data/{stim}_frames/'
save_path = f'../data/{stim}_frames_resnet50_srp_50/'

if not os.path.isdir(save_path):
    os.mkdir(save_path) 
    
#relu, maxpool, block1, block2, block3, block4, avgpool
layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']

fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract, dim_reduction='srp', n_components=50)  #, consolidate_per_layer=False)

In [ ]:


stim='DM'
save_path = f'../data/{stim}_frames_resnet50_srp_50/'

#print the
emb_list = glob.glob(f'{save_path}*.npz')

layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']

print('ResNet50 frame embeddings')
# for emb_f in emb_list:
#     print(emb['frame_0000.jpg'].shape, os.path.splitext(os.path.basename(emb_f))[0])
for layer in layers_to_extract:
    emb = np.load(f'{save_path}{layer}.npz')
    print(emb['frame_0000.jpg'].shape, layer)


### hOW DO THEY DO THIS???
it seems like they apply dimensionality reduction to each image (features) individually... how do they end up with exactly 50

In [ ]:
from sklearn.decomposition import PCA

# Generate a random array of size (1, 2000)
data = np.random.rand(2, 2000)

# Initialize PCA
pca = PCA(n_components=2)

# Fit and transform the data
pca_result = pca.fit_transform(data)

print("PCA Result:\n", pca_result.shape)


## now try to get some additional features! like actual video features
#use conda env net2brain2

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='slowfast_r50', netset='Pyvideo', device='cuda') #cuda worked with 1080ti

In [ ]:
#fx.get_all_layers()
fx.layers_to_extract

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import os

stim='DM'
stimuli_path = f'../data/{stim}_videos/'
save_path = f'../data/{stim}_videos_slowfast_r50/'

if not os.path.isdir(save_path):
    os.mkdir(save_path) 
    
#relu, maxpool, block1, block2, block3, block4, avgpool

fx.extract(data_path=stimuli_path, save_path=save_path)  #, consolidate_per_layer=False)

## try resnet50 with video inputs so they average - maybe better?

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='ResNet50', netset='Standard', device='cuda') #cuda not working now, try dif gpu?

import os

stim='DM'
stimuli_path = f'../data/{stim}_videos/'
save_path = f'../data/{stim}_videos_resnet50/'

if not os.path.isdir(save_path):
    os.mkdir(save_path) 
    
#relu, maxpool, block1, block2, block3, block4, avgpool
layers_to_extract=['relu','maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']

fx.extract(data_path=stimuli_path, save_path=save_path, layers_to_extract=layers_to_extract)  #, consolidate_per_layer=False)

## extract motion features using pymoten

for pymoten, use hbn_asd conda env

#### Despicable Me

In [ ]:
import moten

In [ ]:
import cv2

# Path to the video file
video_path = '../data/DM.mp4'

# Open the video file
video = cv2.VideoCapture(video_path)

# Get the total number of frames
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

print(f'Total number of frames: {total_frames}')

width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'Width: {width}, Height: {height}')

fps = video.get(cv2.CAP_PROP_FPS)

print(f'FPS: {fps}')

# Release the video capture object
video.release()

In [ ]:
video_file = '../data/DM.mp4'
small_vhsize = (72, 108)        # height x width
luminance_images = moten.io.video2luminance(video_file, size=small_vhsize)
# luminance_images.shape
# plt.imshow(luminance_images[0,:,:])
# plt.imshow(luminance_images[99,:,:])

In [ ]:
# eta 4 min
# Create a pyramid of spatio-temporal gabor filters
nimages, vdim, hdim = luminance_images.shape
pyramid = moten.get_default_pyramid(vhsize=(vdim, hdim), fps=30)

# Compute motion energy features
moten_features = pyramid.project_stimulus(luminance_images)

In [ ]:
import h5py
# Path to the HDF5 file
hdf5_path = '../data/features/DM_pymoten.h5'

# Create an HDF5 file
with h5py.File(hdf5_path, 'w') as hdf5_file:
    # Create a dataset in the file
    hdf5_file.create_dataset('pymoten', data=moten_features)

print(f'Motion features saved to {hdf5_path}')


In [ ]:
# # Path to the HDF5 file

# # Open the HDF5 file
# with h5py.File(hdf5_path, 'r') as hdf5_file:
#     # Access the dataset

# # Print the shape of the loaded data
# print(f'Motion features shape: {motion_features.shape}')


#### The Present

In [ ]:
import cv2
# Path to the video file
video_path = '../data/TP.mp4'
# Open the video file
video = cv2.VideoCapture(video_path)
# Get the total number of frames
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Total number of frames: {total_frames}')
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Width: {width}, Height: {height}')
fps = video.get(cv2.CAP_PROP_FPS)
print(f'FPS: {fps}')
# Release the video capture object
video.release()

In [ ]:
stim='TP'
video_file = f'../data/{stim}.mp4'
small_vhsize = (72, 128)        # height x width
luminance_images = moten.io.video2luminance(video_file, size=small_vhsize)
# luminance_images.shape
# plt.imshow(luminance_images[0,:,:])
# plt.imshow(luminance_images[99,:,:])
# eta 4 min
# Create a pyramid of spatio-temporal gabor filters
nimages, vdim, hdim = luminance_images.shape
pyramid = moten.get_default_pyramid(vhsize=(vdim, hdim), fps=24)

# Compute motion energy features
moten_features = pyramid.project_stimulus(luminance_images)

import h5py
# Path to the HDF5 file
hdf5_path = f'../data/features/{stim}_pymoten.h5'

# Create an HDF5 file
with h5py.File(hdf5_path, 'w') as hdf5_file:
    # Create a dataset in the file
    hdf5_file.create_dataset('pymoten', data=moten_features)

print(f'Motion features saved to {hdf5_path}')


#### Friends

In [ ]:
from pathlib import Path
import os
import cv2
# Path to the video file
stim='friends_s01e02a'
video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends.stimuli' / 's1' / f'{stim}.mkv')
# Open the video file
video = cv2.VideoCapture(video_path)
# Get the total number of frames
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Total number of frames: {total_frames}')
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Width: {width}, Height: {height}')
fps = video.get(cv2.CAP_PROP_FPS)
print(f'FPS: {fps}')
# Release the video capture object
video.release()

In [ ]:
video_file = video_path
small_vhsize = (72, 48)        # height x width
luminance_images = moten.io.video2luminance(video_file, size=small_vhsize)
# luminance_images.shape
# plt.imshow(luminance_images[0,:,:])
# plt.imshow(luminance_images[99,:,:])
# eta 4 min
# Create a pyramid of spatio-temporal gabor filters
nimages, vdim, hdim = luminance_images.shape
pyramid = moten.get_default_pyramid(vhsize=(vdim, hdim), fps=30)

# Compute motion energy features
moten_features = pyramid.project_stimulus(luminance_images)

import h5py
# Path to the HDF5 file
hdf5_path = f'../data/features/{stim}_pymoten.h5'

# Create an HDF5 file
with h5py.File(hdf5_path, 'w') as hdf5_file:
    # Create a dataset in the file
    hdf5_file.create_dataset('pymoten', data=moten_features)

print(f'Motion features saved to {hdf5_path}')

In [ ]:
from pathlib import Path
import os
import cv2
stim='friends_s01e02b'

# Path to the video file
video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends.stimuli' / 's1' / f'{stim}.mkv')
# Open the video file
video = cv2.VideoCapture(video_path)
# Get the total number of frames
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Total number of frames: {total_frames}')
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Width: {width}, Height: {height}')
fps = video.get(cv2.CAP_PROP_FPS)
print(f'FPS: {fps}')
# Release the video capture object
video.release()

In [ ]:
video_file = video_path
small_vhsize = (72, 48)        # height x width
luminance_images = moten.io.video2luminance(video_file, size=small_vhsize)
# luminance_images.shape
# plt.imshow(luminance_images[0,:,:])
# plt.imshow(luminance_images[99,:,:])
# eta 4 min
# Create a pyramid of spatio-temporal gabor filters
nimages, vdim, hdim = luminance_images.shape
pyramid = moten.get_default_pyramid(vhsize=(vdim, hdim), fps=30)

# Compute motion energy features
moten_features = pyramid.project_stimulus(luminance_images)

import h5py
# Path to the HDF5 file
hdf5_path = f'../data/features/{stim}_pymoten.h5'

# Create an HDF5 file
with h5py.File(hdf5_path, 'w') as hdf5_file:
    # Create a dataset in the file
    hdf5_file.create_dataset('pymoten', data=moten_features)

print(f'Motion features saved to {hdf5_path}')

## extract low level visual features

### pliers

In [ ]:
from pliers.extractors import VibranceExtractor, SharpnessExtractor, SaliencyExtractor, BrightnessExtractor
from pliers.stimuli import VideoStim

#### brightness

In [ ]:
video = VideoStim('../data/DM.mp4')
extractor = BrightnessExtractor()
brightness = extractor.transform(video)
brightness_values = [result.to_df()['brightness'].values for result in brightness]
brightness_values=resample(np.asanyarray(brightness_values), 750, axis=0)
np.save('../data/features/DM_brightness_pliers.npy', brightness_values)

#### vibrance

In [ ]:
# Load the video as a VideoStim object
video = VideoStim('../data/DM.mp4')
# Create the BrightnessExtractor
extractor = VibranceExtractor()
# Apply the extractor to the video
vibrance = extractor.transform(video)
vibrance_values = [result.to_df()['vibrance'].values for result in vibrance]
np.save('../data/features/DM_vibrance.npy', resample(np.asanyarray(vibrance_values), 750, axis=0))

#### sharpness

In [ ]:
extractor = SharpnessExtractor()
sharpness = extractor.transform(video)
sharpness_values = [result.to_df()['sharpness'].values for result in sharpness]
np.save('../data/features/DM_sharpness.npy', resample(np.asanyarray(sharpness_values), 750, axis=0))

#### saliency

In [ ]:
video = VideoStim('../data/DM.mp4')
extractor = SaliencyExtractor()
saliency = extractor.transform(video)
saliency_values = [result.to_df()['max_saliency'].values for result in saliency]
np.save('../data/features/DM_max_saliency.npy', resample(np.asanyarray(saliency_values), 750, axis=0))
saliency_values = [result.to_df()['frac_high_saliency'].values for result in saliency]
np.save('../data/features/DM_frac_high_saliency.npy', resample(np.asanyarray(saliency_values), 750, axis=0))

### extract brightness, contrast, color myself

In [ ]:
import cv2
video_path = '../data/DM.mp4'

# Open the video file
cap = cv2.VideoCapture(video_path)

brightness_list = []
contrast_list = []
color_features_list = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert the frame to grayscale for brightness and contrast
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Global Brightness: mean pixel value of the grayscale image
    brightness = np.mean(gray_frame)
    brightness_list.append(brightness)
    
    # Global Contrast: standard deviation of pixel values in the grayscale image
    contrast = np.std(gray_frame)
    contrast_list.append(contrast)
    
    # Color Features: mean of RGB channels
    mean_color = cv2.mean(frame)[:3]  # Ignore alpha channel if present
    color_features_list.append(mean_color)
    
    # If you want HSV features instead of RGB:
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mean_hsv = cv2.mean(hsv_frame)[:3]
    # Append mean_hsv to your list if desired

# Release the video capture
cap.release()

# Print or store the extracted features
# print("Brightness values: ", brightness_list)
# print("Contrast values: ", contrast_list)
# print("Color features (mean RGB): ", color_features_list)



# Example usage:

brightness_list=resample(np.asanyarray(brightness_list), 750, axis=0)
contrast_list=resample(np.asanyarray(contrast_list), 750, axis=0)
color_features_list=resample(np.asanyarray(color_features_list), 750, axis=0)

# Save the color features array to .npy
np.save('../data/features/DM_color.npy', color_features_list)
np.save('../data/features/DM_brightness.npy', brightness_list)
np.save('../data/features/DM_contrast.npy', contrast_list)


print("Color features array shape:", color_features_list.shape)


#### also for friends

In [ ]:
from pathlib import Path
import os
import cv2
stim='s01e02b'
video_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends.stimuli' / 's1' / f'friends_{stim}.mkv')

# Open the video file
cap = cv2.VideoCapture(video_path)

brightness_list = []
contrast_list = []
color_features_list = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert the frame to grayscale for brightness and contrast
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Global Brightness: mean pixel value of the grayscale image
    brightness = np.mean(gray_frame)
    brightness_list.append(brightness)
    
    # Global Contrast: standard deviation of pixel values in the grayscale image
    contrast = np.std(gray_frame)
    contrast_list.append(contrast)
    
    # Color Features: mean of RGB channels
    mean_color = cv2.mean(frame)[:3]  # Ignore alpha channel if present
    color_features_list.append(mean_color)
    
    # If you want HSV features instead of RGB:
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mean_hsv = cv2.mean(hsv_frame)[:3]
    # Append mean_hsv to your list if desired

# Release the video capture
cap.release()

# Print or store the extracted features
# print("Brightness values: ", brightness_list)
# print("Contrast values: ", contrast_list)
# print("Color features (mean RGB): ", color_features_list)



# Example usage:

brightness_list=resample(np.asanyarray(brightness_list), 472, axis=0)
contrast_list=resample(np.asanyarray(contrast_list), 472, axis=0)
color_features_list=resample(np.asanyarray(color_features_list), 472, axis=0)

# Save the color features array to .npy
np.save(f'../data/features/friends_{stim}_color.npy', color_features_list)
np.save(f'../data/features/friends_{stim}_brightness.npy', brightness_list)
np.save(f'../data/features/friends_{stim}_contrast.npy', contrast_list)


print("Color features array shape:", color_features_list.shape)


## try alternate brightnesses:

In [ ]:
import cv2

def extract_brightness(video_path):
    # Open video
    video_capture = cv2.VideoCapture(video_path)
    brightness_values = []
    
    while video_capture.isOpened():
        # Read frame-by-frame
        ret, frame = video_capture.read()
        
        if not ret:
            break  # End of video

        # Convert frame to float32 for precision
        
        # Take the maximum value of RGB at each pixel
        max_rgb = np.max(frame, axis=2)
        
        # Compute the mean of the max RGB values (average luminosity)
        brightness = np.mean(max_rgb) / 255.0  # Normalize to [0, 1]
        
        # Append the brightness value for this frame
        brightness_values.append(brightness)
    
    # Release the video capture
    video_capture.release()
    
    return brightness_values

# Example usage
video_path = '../data/DM.mp4'
brightness = extract_brightness(video_path)
#print(brightness)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(brightness)

In [ ]:

# Example usage:
brightness_list_750 = segment_and_average_exact(brightness, num_segments=750)

In [ ]:
np.save('../data/features/DM_brightness2.npy', np.array(brightness_list_750))


In [ ]:
plt.plot(brightness_list_750)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.mean(color_features_array,axis=1))
plt.plot(brightness_array)
plt.plot(contrast_array)

# SO THE MEAN OF COLOR IS REALLY BRIOGHTNESS

#### try perceptual brightness

In [ ]:
import cv2


gamma = 2.2
def gamma_correction(channel):
    return ((channel / 255.0) ** gamma) * 255


# Function to calculate brightness
def calculate_brightness(frame):
    # Split the frame into its Red, Green, and Blue components
    B, G, R = cv2.split(frame)

    # Apply the perceptual brightness formula
    brightness = 0.299 * R + 0.587 * G + 0.114 * B

    # Average the brightness over the whole frame
    return brightness.mean()

def calculate_perceptual_brightness_with_gamma(frame):
    B, G, R = cv2.split(frame)

    # Apply gamma correction
    R = gamma_correction(R)
    G = gamma_correction(G)
    B = gamma_correction(B)

    # Calculate brightness with corrected values
    brightness = 0.299 * R + 0.587 * G + 0.114 * B
    return brightness.mean()

# Open the video file
video = cv2.VideoCapture('../data/DM.mp4')

brightness_values = []
brightness_values4 = []

# Loop through the video frames
while(video.isOpened()):
    ret, frame = video.read()  # Read a frame
    if not ret:
        break
    # Calculate brightness for the current frame
    brightness_value = calculate_brightness(frame)
    brightness_value4 = calculate_perceptual_brightness_with_gamma(frame)
    brightness_values4.append(brightness_value4)
    brightness_values.append(brightness_value)

# Release the video capture object
video.release()


# # Example usage:

brightness_values=resample(np.asanyarray(brightness_values), 750, axis=0)
brightness_values4=resample(np.asanyarray(brightness_values4), 750, axis=0)


# # Now you have the brightness values for each frame
np.save('../data/features/DM_brightness3.npy', np.array(brightness_values))
np.save('../data/features/DM_brightness4.npy', np.array(brightness_values4))



#### also for friends

In [ ]:
import cv2

stim='s01e02b'
gamma = 2.2
def gamma_correction(channel):
    return ((channel / 255.0) ** gamma) * 255


# Function to calculate brightness
def calculate_brightness(frame):
    # Split the frame into its Red, Green, and Blue components
    B, G, R = cv2.split(frame)

    # Apply the perceptual brightness formula
    brightness = 0.299 * R + 0.587 * G + 0.114 * B

    # Average the brightness over the whole frame
    return brightness.mean()

def calculate_perceptual_brightness_with_gamma(frame):
    B, G, R = cv2.split(frame)

    # Apply gamma correction
    R = gamma_correction(R)
    G = gamma_correction(G)
    B = gamma_correction(B)

    # Calculate brightness with corrected values
    brightness = 0.299 * R + 0.587 * G + 0.114 * B
    return brightness.mean()

# Open the video file
video = cv2.VideoCapture(str(STIMULI / 'friends.stimuli' / 's1' / f'friends_{stim}.mkv'))

brightness_values = []
brightness_values4 = []

# Loop through the video frames
while(video.isOpened()):
    ret, frame = video.read()  # Read a frame
    if not ret:
        break
    # Calculate brightness for the current frame
    brightness_value = calculate_brightness(frame)
    brightness_value4 = calculate_perceptual_brightness_with_gamma(frame)
    brightness_values4.append(brightness_value4)
    brightness_values.append(brightness_value)

# Release the video capture object
video.release()

# # Example usage:

brightness_values=resample(np.asanyarray(brightness_values), 472, axis=0)
brightness_values4=resample(np.asanyarray(brightness_values4), 472, axis=0)

# # Now you have the brightness values for each frame
np.save(f'../data/features/friends_{stim}_brightness3.npy', np.array(brightness_values))
np.save(f'../data/features/friends_{stim}_brightness4.npy', np.array(brightness_values4))

#### get center biased brightness

In [ ]:
import cv2

def create_gaussian_mask(shape, sigma=0.4):
    """ Create a 2D Gaussian weighting mask with higher weights in the center """
    h, w = shape[:2]
    # Create coordinate grid
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    x, y = np.meshgrid(x, y)

    # Calculate Gaussian weights based on distance from the center
    d = np.sqrt(x**2 + y**2)
    gaussian_mask = np.exp(-(d**2) / (2.0 * sigma**2))
    
    # Normalize the mask to sum to 1
    gaussian_mask /= np.sum(gaussian_mask)
    
    return gaussian_mask

def calculate_center_biased_brightness(frame, gaussian_mask):
    """ Calculate center-biased brightness for a given frame using a Gaussian mask """
    # Convert frame to grayscale to compute brightness (if needed)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Normalize the brightness values to a range of 0 to 1
    normalized_brightness = gray_frame / 255.0
    
    # Apply the Gaussian mask to compute center-biased brightness
    center_biased_brightness = np.sum(normalized_brightness * gaussian_mask)
    
    return center_biased_brightness

def extract_center_biased_brightness(video_path):
    # Open the video file
    video = cv2.VideoCapture(video_path)
    
    # Get the shape of the first frame to create a Gaussian mask
    ret, frame = video.read()
    if not ret:
        raise ValueError("Could not read video.")
    
    # Create the Gaussian mask for center bias (based on frame size)
    gaussian_mask = create_gaussian_mask(frame.shape[:2])
    
    # Reset video to the start
    video.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    center_biased_brightness_values = []
    
    # Loop through the video frames and calculate center-biased brightness
    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break
        
        # Calculate center-biased perceptual brightness for each frame
        brightness_value = calculate_center_biased_brightness(frame, gaussian_mask)
        center_biased_brightness_values.append(brightness_value)
    
    video.release()
    
    # Convert the list to a NumPy array
    return np.array(center_biased_brightness_values)

# Example: Extract center-biased brightness from a video
video_path = '../data/DM.mp4'
brightness4_center_biased = extract_center_biased_brightness(video_path)

brightness4_center_biased=resample(np.asanyarray(brightness4_center_biased), 750, axis=0)

np.save('../data/features/DM_brightness_perceptual_center.npy', np.array(brightness4_center_biased))

#### visualize the gaussian for center bias

In [ ]:
def create_gaussian_mask(shape, sigma):
    """ Create a 2D Gaussian weighting mask with higher weights in the center """
    h, w = shape[:2]
    # Create coordinate grid
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    x, y = np.meshgrid(x, y)

    # Calculate Gaussian weights based on distance from the center
    d = np.sqrt(x**2 + y**2)
    gaussian_mask = np.exp(-(d**2) / (2.0 * sigma**2))
    
    # Normalize the mask to sum to 1
    gaussian_mask /= np.sum(gaussian_mask)
    
    return gaussian_mask
    
    
video = cv2.VideoCapture(video_path)

# Get the shape of the first frame to create a Gaussian mask
ret, frame = video.read()
if not ret:
    raise ValueError("Could not read video.")

# Create the Gaussian mask for center bias (based on frame size)
gaussian_mask = create_gaussian_mask(frame.shape[:2],0.4)
plt.imshow(gaussian_mask, cmap='hot', interpolation='nearest')

#### extract color

In [ ]:
import cv2

def extract_lab_color_features(frame):
    # Convert the frame from BGR to LAB color space
    lab_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)

    # Split the LAB frame into L, A, and B channels
    L, A, B = cv2.split(lab_frame)

    # Calculate mean and variance for A (green-red) and B (blue-yellow) channels
    a_mean = np.mean(A)
    a_variance = np.var(A)
    
    b_mean = np.mean(B)
    b_variance = np.var(B)
    
    # Return a 4-dimensional feature vector
    return [a_mean, a_variance, b_mean, b_variance]

# Example usage for a video
def process_video(video_path):
    # Open the video file
    video = cv2.VideoCapture(video_path)
    color_features = []

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break
        
        # Extract LAB color features from the frame
        lab_features = extract_lab_color_features(frame)
        color_features.append(lab_features)

    video.release()
    return np.array(color_features)

# Example: Process a video and get the LAB color features
video_path = '../data/DM.mp4'
lab_color_features = process_video(video_path)

lab_color_features=resample(np.asanyarray(lab_color_features), 750, axis=0)

np.save('../data/features/DM_lab_color4.npy', np.array(lab_color_features))


### extract gabor filters

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_gabor_filters(orientations, frequencies, ksize=31):
    """ Create a list of Gabor filters with different orientations and frequencies. """
    filters = []
    for theta in orientations:
        for freq in frequencies:
            # Create the Gabor filter with the specific orientation and frequency
            kernel = cv2.getGaborKernel((ksize, ksize), 4.0, theta, freq, 0.5, 0, ktype=cv2.CV_32F)
            filters.append(kernel)
    return filters

def apply_gabor_filters(frame, filters):
    """ Apply a list of Gabor filters to a single frame and return the filter responses. """
    responses = []
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert frame to grayscale
    
    for kernel in filters:
        # Apply each Gabor filter to the frame
        filtered_image = cv2.filter2D(gray_frame, cv2.CV_8UC3, kernel)
        responses.append(filtered_image)
    
    return np.array(responses)

def extract_gabor_responses_from_video(video_path, orientations, frequencies, ksize=31):
    """ Extract Gabor filter responses from each frame of the video. """
    video = cv2.VideoCapture(video_path)
    
    # Create the Gabor filters
    filters = create_gabor_filters(orientations, frequencies, ksize)
    
    gabor_responses = []
    
    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break
        
        # Apply Gabor filters and get responses for the frame
        frame_responses = apply_gabor_filters(frame, filters)
        gabor_responses.append(frame_responses)
    
    video.release()
    
    return np.array(gabor_responses)

# Example usage:
orientations = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]  # 0, 45, 90, 135 degrees
frequencies = [0.05, 0.1, 0.2]  # Spatial frequencies
video_path = '../data/DM.mp4'

# Extract Gabor filter responses
gabor_responses = extract_gabor_responses_from_video(video_path, orientations, frequencies)

# Example: Plot the response of the first frame for the first Gabor filter
plt.imshow(gabor_responses[0, 0], cmap='gray')
plt.title('Gabor Filter Response (First Frame, First Filter)')
plt.show()
